# Enhancing Personal Assistant Application with Tools and Function Calling Capabilities

Enhance the agentic AI application with structured tool definitions using JSON Schema and LLM-native function calling.

## Setup and Imports

In [1]:
!pip install litellm python-dotenv

In [2]:
import os
import json
from typing import List, Dict, Tuple
import time
from dotenv import load_dotenv
from litellm import completion

# Load secrets from .env
load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError(
        "GOOGLE_API_KEY is missing. Create a .env file with GOOGLE_API_KEY=your_key"
    )

def generate_response(messages: List[Dict], tools: List[Dict] = None) -> object:
    """Call Gemini API with retry logic for rate limits."""
    max_retries = 5
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": "gemini/gemini-3-flash-preview",
                "messages": messages,
                "max_tokens": 512,
                "temperature": 0.7
            }
            if tools:
                kwargs["tools"] = tools
            
            return completion(**kwargs)
        except Exception as e:
            error_str = str(e).lower()
            if "rate_limit" in error_str or "quota" in error_str or "429" in error_str:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) * 2  # Exponential backoff: 2, 4, 8, 16, 32 seconds
                    print(f"Rate limited. Waiting {wait_time} seconds (attempt {attempt + 1}/{max_retries})...")
                    time.sleep(wait_time)
                    continue
            raise


## Tool Functions

In [3]:
# Tool implementations
def create_reminder(reminder_text: str, priority: str = "normal") -> dict:
    """Create a reminder for the user with optional priority level."""
    return {
        "status": "created",
        "reminder": reminder_text,
        "priority": priority,
        "message": f"Reminder set ({priority}): {reminder_text}"
    }

def add_task(task: str, category: str = "general") -> dict:
    """Add a task to the todo list with optional category."""
    return {
        "status": "success",
        "task": task,
        "category": category,
        "message": f"Task added ({category}): {task}"
    }

def take_note(note_content: str, tags: str = "") -> dict:
    """Take a note and save it with optional tags."""
    return {
        "status": "saved",
        "note": note_content,
        "tags": tags.split(",") if tags else [],
        "message": f"Note saved: {note_content[:50]}..."
    }

def schedule_event(event_name: str, date: str, time: str) -> dict:
    """Schedule an event with date and time."""
    return {
        "status": "scheduled",
        "event": event_name,
        "date": date,
        "time": time,
        "message": f"Event scheduled: {event_name} on {date} at {time}"
    }

def analyze_text(text: str, language: str = "en", detailed: bool = False) -> dict:
    """Analyzes text for sentiment and extracts keywords (NEW TOOL)."""
    words = text.lower().split()
    keywords = [w for w in words if len(w) > 4]
    
    return {
        "status": "analyzed",
        "text_length": len(text),
        "word_count": len(words),
        "language": language,
        "keywords": keywords[:5] if detailed else keywords[:3],
        "message": f"Text analyzed: {len(words)} words"
    }

# Map tool names to functions
tool_functions = {
    "create_reminder": create_reminder,
    "add_task": add_task,
    "take_note": take_note,
    "schedule_event": schedule_event,
    "analyze_text": analyze_text
}

print(" Tool functions defined")

 Tool functions defined


## Tool Schemas (JSON Schema Format)

In [4]:
# JSON Schema tool definitions for native function calling
tools = [
    {
        "type": "function",
        "function": {
            "name": "create_reminder",
            "description": "Create a reminder with optional priority",
            "parameters": {
                "type": "object",
                "properties": {
                    "reminder_text": {"type": "string", "description": "Reminder text"},
                    "priority": {"type": "string", "enum": ["low", "normal", "high"], "description": "Priority level"}
                },
                "required": ["reminder_text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_task",
            "description": "Add a task with optional category",
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {"type": "string", "description": "Task description"},
                    "category": {"type": "string", "description": "Task category"}
                },
                "required": ["task"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "take_note",
            "description": "Take a note with optional tags",
            "parameters": {
                "type": "object",
                "properties": {
                    "note_content": {"type": "string", "description": "Note content"},
                    "tags": {"type": "string", "description": "Comma-separated tags"}
                },
                "required": ["note_content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "schedule_event",
            "description": "Schedule an event with date and time",
            "parameters": {
                "type": "object",
                "properties": {
                    "event_name": {"type": "string", "description": "Event name"},
                    "date": {"type": "string", "description": "Event date (YYYY-MM-DD)"},
                    "time": {"type": "string", "description": "Event time (HH:MM)"}
                },
                "required": ["event_name", "date", "time"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_text",
            "description": "Analyze text for keywords (NEW: multi-parameter tool)",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Text to analyze"},
                    "language": {"type": "string", "enum": ["en", "es", "fr", "de"], "description": "Language"},
                    "detailed": {"type": "boolean", "description": "Detailed analysis"}
                },
                "required": ["text"]
            }
        }
    }
]

# Agent system prompt
agent_rules = [
    {"role": "system", "content": "You are a helpful Personal Assistant AI. Use available tools to help users."}
]


## Native Function Calling with Error Handling

In [5]:
def extract_tool_call(response) -> Tuple[str, dict]:
    """Extract tool call from LLM response using native function calling."""
    try:
        if not hasattr(response, 'choices') or len(response.choices) == 0:
            print(f"No choices in response")
            return None, None
        
        choice = response.choices[0]
        if not hasattr(choice, 'message'):
            print(f"No message in choice")
            return None, None
        
        if not hasattr(choice.message, 'tool_calls') or not choice.message.tool_calls:
            print(f"No tool_calls in message")
            return None, None
        
        tool_call = choice.message.tool_calls[0]
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        return tool_name, tool_args
    except Exception as e:
        print(f"Tool extraction error: {str(e)}")
        return None, None

def validate_args(tool_name: str, args: dict) -> Tuple[bool, str]:
    """Validate tool arguments before execution."""
    try:
        tool_def = next((t for t in tools if t["function"]["name"] == tool_name), None)
        if not tool_def:
            return False, f"Tool '{tool_name}' not found"
        
        required = tool_def["function"]["parameters"].get("required", [])
        for req_param in required:
            if req_param not in args:
                return False, f"Missing required parameter: {req_param}"
        
        return True, "Valid"
    except Exception as e:
        return False, f"Validation error: {str(e)}"


## Agent Loop with Native Function Calling

In [6]:
def run_agent(user_messages: List[str], delay_between_calls: float = 2.0) -> List[Dict]:
    """Run agent using native function calling with error handling."""
    memory = agent_rules.copy()
    conversation_log = []
    
    for idx, user_input in enumerate(user_messages):
        if not user_input.strip():
            continue
        
        # Add delay between consecutive API calls to avoid rate limiting
        if idx > 0:
            print(f"Waiting {delay_between_calls}s to avoid rate limiting...")
            time.sleep(delay_between_calls)
            
        print(f"\nUSER: {user_input}")
        memory.append({"role": "user", "content": user_input})
        
        try:
            response = generate_response(memory, tools=tools)
            tool_name, tool_args = extract_tool_call(response)
            
            if not tool_name:
                print("No tool call found")
                continue
            
            print(f"TOOL: {tool_name} | ARGS: {tool_args}")
            
            if tool_name not in tool_functions:
                result = {"error": f"Unknown tool: {tool_name}"}
            else:
                is_valid, msg = validate_args(tool_name, tool_args)
                if not is_valid:
                    result = {"error": msg}
                else:
                    try:
                        result = tool_functions[tool_name](**tool_args)
                        print(f"✓ Result: {result['message']}")
                    except TypeError as e:
                        result = {"error": f"Invalid arguments: {str(e)}"}
            
            memory.append({"role": "assistant", "content": f"Tool executed: {json.dumps(result)}"})
            conversation_log.append({
                "user": user_input,
                "tool": tool_name,
                "args": tool_args,
                "result": result
            })
            
        except Exception as e:
            print(f"Error: {str(e)}")
            conversation_log.append({"user": user_input, "error": str(e)})
    
    return conversation_log


## Comprehensive Test Suite

In [7]:
# Run all test cases
test_cases = {
    "Test 1: Reminders with Priority": [
        "I need to remember to call the dentist tomorrow at 2 PM",
        "Can you also remind me with high priority to prepare presentation slides?"
    ],
    "Test 2: Tasks with Categories": [
        "Add a work task: Complete project report",
        "Add a personal task: Buy groceries"
    ],
    "Test 3: Notes with Tags": [
        "Take a note on Q1 budget meeting, tag as finance,meeting",
        "Add another note about project deadline March 15th, tag as deadline"
    ],
    "Test 4: Events with Date/Time": [
        "Schedule a team standup on 2026-02-17 at 10:00",
        "Schedule doctor's appointment on 2026-02-19 at 15:00"
    ],
    "Test 5: Text Analysis (Multi-Parameter NEW Tool)": [
        "Analyze this: class Dog with methods init and bark for sounds",
        "Analyze in detailed mode in English"
    ],
    "Test 6: Error Handling": [
        "I need to remember to submit timesheet",
        "Add a work task for the project"
    ]
}

results = {}

for test_name, inputs in test_cases.items():
    print(f"\n{test_name}")
    results[test_name] = run_agent(inputs)



Test 1: Reminders with Priority

USER: I need to remember to call the dentist tomorrow at 2 PM
TOOL: schedule_event | ARGS: {'time': '14:00', 'date': '2025-05-23', 'event_name': 'Call the dentist'}
✓ Result: Event scheduled: Call the dentist on 2025-05-23 at 14:00
Waiting 2.0s to avoid rate limiting...

USER: Can you also remind me with high priority to prepare presentation slides?
TOOL: create_reminder | ARGS: {'priority': 'high', 'reminder_text': 'Prepare presentation slides'}
✓ Result: Reminder set (high): Prepare presentation slides

Test 2: Tasks with Categories

USER: Add a work task: Complete project report
TOOL: add_task | ARGS: {'category': 'work', 'task': 'Complete project report'}
✓ Result: Task added (work): Complete project report
Waiting 2.0s to avoid rate limiting...

USER: Add a personal task: Buy groceries
TOOL: add_task | ARGS: {'category': 'personal', 'task': 'Buy groceries'}
✓ Result: Task added (personal): Buy groceries

Test 3: Notes with Tags

USER: Take a note 

In [8]:

# Conversation where the user interacts with multiple tools
multi_tool_conversation = [
    "I need to plan my week. First, remind me to attend the project review meeting with high priority",
    "Schedule that meeting for 2026-02-17 at 14:00",
    "Add a task to prepare presentation materials before the meeting",
    "Take a note: Discussion topics - Q1 roadmap, team capacity, technical debt",
    "Analyze this text for keywords: 'machine learning optimization algorithms neural networks deep learning'",
    "Also schedule a lunch meeting with the team on 2026-02-18 at 12:00"
]

print("\n Conversation Sequence:")
for i, msg in enumerate(multi_tool_conversation, 1):
    print(f"Message {i}: {msg[:65]}...")

print("\n")
print("EXECUTING CONVERSATION WITH MEMORY TRACKING")

# Run the conversation
multi_tool_results = run_agent(multi_tool_conversation)

print("\n")
print("MEMORY PERSISTENCE & TOOL ANALYSIS")

# Extract and display memory state
print(f"\n Conversation completed with {len(multi_tool_results)} tool calls")
print("\nTools used in this conversation:")

unique_tools = set()
for i, result in enumerate(multi_tool_results, 1):
    tool_name = result.get("tool", "N/A")
    if tool_name not in unique_tools:
        unique_tools.add(tool_name)
    print(f"  {i}. {tool_name}")

print(f"\n Total unique tools used: {len(unique_tools)}")
print(f"Tools: {', '.join(sorted(unique_tools))}")
print(f"Total interactions: {len(multi_tool_results)}")

print("\n")
print("DETAILED EXECUTION RESULTS (Memory Persistence)")

# Display detailed results highlighting tool calls and results
for i, result in enumerate(multi_tool_results, 1):
    print(f"\n[Interaction {i}]")
    print(f"User Input:     {result.get('user', 'N/A')[:50]}...")
    print(f"Tool Called:    {result.get('tool', 'Error')}")
    
    if "args" in result and result.get("args"):
        args_str = str(result.get('args'))
        print(f"Arguments:      {args_str[:70]}...")
    
    if "result" in result:
        result_dict = result.get('result', {})
        if "error" in result_dict:
            print(f"Status:         ✗ ERROR - {result_dict['error']}")
        else:
            msg = result_dict.get('message', 'Executed')
            print(f"Status:         ✓ SUCCESS")
            print(f"Result:         {msg[:65]}...")


 Conversation Sequence:
Message 1: I need to plan my week. First, remind me to attend the project re...
Message 2: Schedule that meeting for 2026-02-17 at 14:00...
Message 3: Add a task to prepare presentation materials before the meeting...
Message 4: Take a note: Discussion topics - Q1 roadmap, team capacity, techn...
Message 5: Analyze this text for keywords: 'machine learning optimization al...
Message 6: Also schedule a lunch meeting with the team on 2026-02-18 at 12:0...


EXECUTING CONVERSATION WITH MEMORY TRACKING

USER: I need to plan my week. First, remind me to attend the project review meeting with high priority
TOOL: create_reminder | ARGS: {'reminder_text': 'Attend the project review meeting', 'priority': 'high'}
✓ Result: Reminder set (high): Attend the project review meeting
Waiting 2.0s to avoid rate limiting...

USER: Schedule that meeting for 2026-02-17 at 14:00
TOOL: schedule_event | ARGS: {'date': '2026-02-17', 'event_name': 'Project review meeting', 'time': '14:0

In [9]:
# Display Tool Call Structure and Schema Validation
print("\n")
print("STRUCTURED TOOL CALL DEMONSTRATION")

print("\n Tool Definition Example (Create Reminder):")

reminder_tool = tools[0]  # First tool
print(f"Tool Name: {reminder_tool['function']['name']}")
print(f"Description: {reminder_tool['function']['description']}")
print(f"\nParameters:")
params = reminder_tool['function']['parameters']
for prop_name, prop_details in params['properties'].items():
    prop_type = prop_details.get('type', 'string')
    required = "Required" if prop_name in params.get('required', []) else " Optional"
    if 'enum' in prop_details:
        print(f"  - {prop_name} ({prop_type}): {required}")
        print(f"    Allowed values: {prop_details['enum']}")
    else:
        print(f"  - {prop_name} ({prop_type}): {required}")
        print(f"    Description: {prop_details.get('description', 'N/A')}")

print("\n Example Tool Execution Flow:")

# Show what happens when a tool is called
example_args = {"reminder_text": "Project review meeting", "priority": "high"}
print(f"\n1. LLM generates structured call:")
print(f"   Tool: create_reminder")
print(f"   Args: {example_args}")

# Validate
is_valid, msg = validate_args("create_reminder", example_args)
print(f"\n2. Validation:")
print(f"   Status: {'✓ PASSED' if is_valid else '✗ FAILED'}")
print(f"   Message: {msg}")

# Execute
if is_valid:
    result = create_reminder(**example_args)
    print(f"\n3. Execution Result:")
    for key, value in result.items():
        print(f"   {key}: {value}")


print("\nMemory Architecture:")
print(f"Agent Rules (System): {len(agent_rules)} message(s)")
print(f"Tool Functions Available: {len(tool_functions)} functions")
print(f"Tool Schemas Defined: {len(tools)} tools with JSON Schema")





STRUCTURED TOOL CALL DEMONSTRATION

 Tool Definition Example (Create Reminder):
Tool Name: create_reminder
Description: Create a reminder with optional priority

Parameters:
  - reminder_text (string): Required
    Description: Reminder text
  - priority (string):  Optional
    Allowed values: ['low', 'normal', 'high']

 Example Tool Execution Flow:

1. LLM generates structured call:
   Tool: create_reminder
   Args: {'reminder_text': 'Project review meeting', 'priority': 'high'}

2. Validation:
   Status: ✓ PASSED
   Message: Valid

3. Execution Result:
   status: created
   reminder: Project review meeting
   priority: high
   message: Reminder set (high): Project review meeting

Memory Architecture:
Agent Rules (System): 1 message(s)
Tool Functions Available: 5 functions
Tool Schemas Defined: 5 tools with JSON Schema
